In [0]:
%pip install -q duckdb google-genai
dbutils.library.restartPython()

In [0]:
import sys, os, importlib, json
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_dir = "/Workspace" + os.path.dirname(nb_path)
sys.path.insert(0, nb_dir)
import tools, agent, eval_set
for m in (tools, agent, eval_set):
    importlib.reload(m)

advisor = agent.Advisor(api_key=dbutils.secrets.get("dubai-agent", "gemini"),
                        data_url=dbutils.secrets.get("dubai-agent", "marts_par"))
print(f"Data as of {advisor.as_of} | model {advisor.model} | {len(eval_set.CASES)} cases\n")

df = eval_set.run(advisor)
overall, by_kind = eval_set.summarize(df)

print("\n=== OVERALL ===")
print(json.dumps(overall, indent=1))
print("\n=== BY QUESTION TYPE ===")
print(by_kind.to_string())
print("\n=== FAILURES ===")
for _, row in df[df.status.isin(["fail", "error"])].iterrows():
    print(f"\n{row.id} [{row.kind}] {row.question}\n  failed: {row.failed_checks}\n  expected: {row.get('expected', '')}"
          f"\n  tools: {row.get('tools_called', '')}\n  unverified: {row.get('unverified_numbers', '')}"
          f"\n  answer: {str(row.get('answer', ''))[:600]}")

df.to_csv(f"{nb_dir}/eval_results_{advisor.as_of}.csv", index=False)
print(f"\nSaved {nb_dir}/eval_results_{advisor.as_of}.csv")

In [0]:
import sys, os, importlib, json
import pandas as pd
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_dir = "/Workspace" + os.path.dirname(nb_path)
sys.path.insert(0, nb_dir)
import tools, agent, eval_set
for m in (tools, agent, eval_set):
    importlib.reload(m)

saved = pd.read_csv(f"{nb_dir}/eval_results_2026-10-14.csv")
print("=== R05 full answer (from the first run) ===")
print(saved.loc[saved.id == "R05", "answer"].iloc[0])

advisor = agent.Advisor(api_key=dbutils.secrets.get("dubai-agent", "gemini"),
                        data_url=dbutils.secrets.get("dubai-agent", "marts_par"))
print("\n=== R05 tool result (top 5) ===")
print(json.dumps(tools.find_segments(5_000_000, "live", limit=5), indent=1)[:2500])

print("\n=== Rerun of the three failed cases ===")
rerun = eval_set.run(advisor, cases=[c for c in eval_set.CASES if c["id"] in {"R05", "M04", "M05"}])
for _, row in rerun.iterrows():
    print(f"\n{row.id}: {row.status.upper()}  failed: {row.failed_checks or '-'}  unverified: {row.unverified_numbers or '-'}")

In [0]:
import sys, os, importlib
import pandas as pd
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_dir = "/Workspace" + os.path.dirname(nb_path)
sys.path.insert(0, nb_dir)
import tools, agent, eval_set
for m in (tools, agent, eval_set):
    importlib.reload(m)

advisor = agent.Advisor(api_key=dbutils.secrets.get("dubai-agent", "gemini"),
                        data_url=dbutils.secrets.get("dubai-agent", "marts_par"))
path = f"{nb_dir}/eval_results_{advisor.as_of}.csv"
df = pd.read_csv(path)
df["attempt"] = "first run"
retry = eval_set.run(advisor, cases=[c for c in eval_set.CASES if c["id"] == "F06"])
retry["attempt"] = "retry after server error"
out = pd.concat([df, retry], ignore_index=True)
out.to_csv(path, index=False)
print(retry[["id", "status", "failed_checks", "seconds", "unverified_numbers"]].to_string(index=False))
print(f"\nSaved {path} ({len(out)} rows)")